# 01. 그래프 데이터 다루기 — PyTorch Geometric

**그래프 신경망(GNN)** 은 노드와 엣지로 이루어진 그래프 데이터를 다룹니다 (예: 소셜 네트워크, 분자 구조, 인용 관계).
`PyTorch Geometric(PyG)` 으로 그래프를 표현하고 시각화하며, 메시지 패싱의 기본을 익힙니다.

1. 그래프 데이터 구조 (노드 특징·엣지)
2. networkx로 시각화
3. GCN 레이어의 메시지 패싱 이해

In [ ]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import networkx as nx
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
import torch_geometric
print('PyG', torch_geometric.__version__)

## 1. 그래프 데이터 구조

PyG는 그래프를 `Data` 객체로 표현합니다.
- **x**: 노드 특징 행렬 (노드 수 × 특징 수)
- **edge_index**: 엣지 연결 (2 × 엣지 수). 각 열이 (출발 노드, 도착 노드)
- **y**: 노드 라벨 (선택)

In [ ]:
# 4개 노드, 양방향 엣지로 연결된 작은 그래프
edge_index = torch.tensor([[0, 1, 1, 2, 2, 3],
                           [1, 0, 2, 1, 3, 2]], dtype=torch.long)
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])   # 노드당 특징 1개
y = torch.tensor([0, 0, 1, 1])                    # 노드 라벨

data = Data(x=x, edge_index=edge_index, y=y)
print(data)
print('노드 수:', data.num_nodes, '| 엣지 수:', data.num_edges)
print('무방향 그래프인가:', data.is_undirected())

## 2. networkx로 시각화

PyG 그래프를 networkx로 변환해 시각화합니다.

In [ ]:
from torch_geometric.utils import to_networkx
G = to_networkx(data, to_undirected=True)

plt.figure(figsize=(5, 4))
nx.draw(G, with_labels=True, node_color=['#4c72b0','#4c72b0','#dd8452','#dd8452'],
        node_size=800, font_color='white', font_size=14)
plt.title('작은 그래프 (색=라벨)')
plt.show()

## 3. GCN 레이어 — 메시지 패싱

GCN(Graph Convolution)은 각 노드가 **이웃 노드의 정보를 모아(aggregation)** 자신의 표현을 갱신합니다.
GCNConv 레이어를 한 번 통과시켜 노드 특징이 이웃 정보로 바뀌는 것을 봅니다.

In [ ]:
conv = GCNConv(in_channels=1, out_channels=2)
out = conv(data.x, data.edge_index)
print('입력 노드 특징 shape:', data.x.shape)
print('GCN 통과 후 shape:', out.shape)
print('\n각 노드는 이제 이웃 정보가 섞인 새 표현을 가집니다:')
print(out.detach())

### 정리

- 그래프를 PyG `Data`(노드 특징·엣지)로 표현하고 networkx로 시각화했습니다.
- GCN 레이어가 이웃 정보를 모아 노드 표현을 갱신하는 메시지 패싱을 확인했습니다.
- 다음 노트북에서는 실제 인용 네트워크(Cora)에서 노드를 분류합니다.